# 10 — Silver Business Rules: streets_business

In [0]:
# Business rules applied here (Day 5 scope, PDF: "currency normalization,
# date standardization" — VStone has no currency, so the equivalent rules
# are the raining range fix and explicit date-part standardization):
#   1. raining clipped to [0,100] — justified by real profiling: 49.29% of
#      all 87.8M rows are negative, symmetric noise around 0, overshooting
#      both bounds (~-1 to ~101). Original value is KEPT, not overwritten —
#      raining_clipped is a new column, so nothing is silently discarded.
#   2. reading_ts split into reading_date / reading_hour — Day 4 already
#      standardized the raw string into a proper TIMESTAMP; this adds the
#      derived parts Gold-layer aggregation (Day 6-7) will want.

from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("silver_schema", "silver", "2. Silver Schema")

CATALOG = dbutils.widgets.get("catalog_name")
SILVER = dbutils.widgets.get("silver_schema")
SOURCE_TABLE = f"{CATALOG}.{SILVER}.streets_silver"
TARGET_TABLE = f"{CATALOG}.{SILVER}.streets_business"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

 Read Silver (batch, not streaming — this is a one-shot materialization, rerun as often as needed since the write below is a full overwrite)

In [0]:
df_silver = spark.table(SOURCE_TABLE)
source_count = df_silver.count()
print(f"Source rows: {source_count:,}")

## Apply business rules

In [0]:
df_business = (df_silver
    .withColumn("raining_clipped",
        F.when(F.col("raining") < 0, F.lit(0.0))
         .when(F.col("raining") > 100, F.lit(100.0))
         .otherwise(F.col("raining")))
    .withColumn("reading_date", F.to_date(F.col("reading_ts")))
    .withColumn("reading_hour", F.hour(F.col("reading_ts")))
    .withColumn("business_load_dt", F.current_timestamp()))

clipped_count = df_business.filter(F.col("raining") != F.col("raining_clipped")).count()
print(f"Rows where raining was clipped: {clipped_count:,} "
      f"({round(clipped_count / source_count * 100, 2)}%)")



## Write as a plain managed Delta table (full overwrite — idempotent)

In [0]:
(df_business.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE))

spark.sql(f"""
COMMENT ON TABLE {TARGET_TABLE} IS
'Silver business rules applied to streets_silver: raining clipped to [0,100] '
'(original preserved in "raining"), reading_ts split into reading_date/reading_hour. '
'Plain Delta table (not DLT-managed) — supports UPDATE/MERGE/time travel, see '
'12_delta_acid_timetravel_demo.py.'
""")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN raining COMMENT 'Original cast value, NOT clipped — may be outside [0,100]'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN raining_clipped COMMENT 'raining clamped to [0,100] — the business-rule value to use downstream'")


## Verification

In [0]:
df_check = spark.table(TARGET_TABLE)
target_count = df_check.count()
out_of_range_after_clip = df_check.filter("raining_clipped < 0 OR raining_clipped > 100").count()

print(f"\n{'='*60}\n  BUSINESS RULES SUMMARY\n{'='*60}")
print(f"  Target table              : {TARGET_TABLE}")
print(f"  Rows                      : {target_count:,}  (source: {source_count:,})")
print(f"  Rows clipped              : {clipped_count:,}")
print(f"  Out-of-range after clip   : {out_of_range_after_clip:,}   (must be 0)")
print(f"{'='*60}")

if target_count != source_count:
    raise Exception(f"Row count mismatch: {target_count:,} vs source {source_count:,}")
if out_of_range_after_clip > 0:
    raise Exception(f"Clip rule failed — {out_of_range_after_clip} rows still out of [0,100].")
print("  ALL CHECKS PASSED")

display(df_check.select("street_id", "reading_ts", "raining", "raining_clipped", "reading_date", "reading_hour").limit(10))
